# Taller 2 · Fase 3: el RAG de EcoMarket, paso a paso

Recorrido ejecutado de la cadena completa: **indexación → consulta con el contexto a la vista →
abstención → integración con el Taller 1 → resumen de los experimentos**.

> **Modo de ejecución.** La primera celda de código detecta qué hay disponible. Si están las claves de
> la configuración activa (`RAG_CONFIG`) y Postgres responde, usa los modelos y la base reales. Si no,
> usa la configuración **`pruebas`**: embeddings léxicos falsos, un LLM falso que copia la primera
> frase del mejor fragmento y una base vectorial **en memoria**. En ese modo la mecánica es la misma,
> pero **las respuestas no son de un modelo real**: los resultados reales están en `outputs/`.

In [1]:
import os

from genai_common.configuraciones import resolver_configuracion
from taller02_rag import atencion, chains, componentes, evaluation, indexing, loaders, servicios


def modo_disponible():
    try:
        config = resolver_configuracion()
    except Exception as exc:
        return None, f"configuración no válida: {exc}"
    faltan = sorted({p.name for p in (config.embeddings, config.llm) if not p.key_configured})
    if faltan:
        return None, f"faltan las claves de {', '.join(faltan)}"
    if not servicios.estado_postgres().ok:
        return None, "Postgres no responde"
    return config, "ok"


config, motivo = modo_disponible()
if config is None:
    os.environ["RAG_CONFIG"] = "pruebas"
    BACKEND = "memoria"
    print(f"Modo PRUEBAS (sin red): {motivo}. Las respuestas son de un modelo falso.")
else:
    BACKEND = "pgvector"
    print(f"Modo REAL: configuración {config.nombre}, tabla {config.tabla}.")
config = resolver_configuracion()
print(f"Embeddings: {config.embeddings.name} ({config.embeddings.model}) · LLM: {config.llm.name}")

Modo PRUEBAS (sin red): faltan las claves de gemini. Las respuestas son de un modelo falso.
Embeddings: fake (lexico-hash) · LLM: fake


## 1. Indexación

Cuatro documentos en cuatro formatos, cada uno con su *loader* y su estrategia de chunking.

In [2]:
from collections import Counter

documentos = loaders.cargar_todo()
chunks = indexing.preparar_chunks("por_tipo")
print("Documentos cargados por tipo:", dict(Counter(d.metadata["tipo"] for d in documentos)))
print("Fragmentos por tipo:          ", dict(Counter(c.metadata["tipo"] for c in chunks)))
print("Preguntas frecuentes (un fragmento cada una):", sum(c.metadata["es_faq"] for c in chunks))

Documentos cargados por tipo: {'politicas': 1, 'credito': 18, 'producto': 15, 'terminos': 2}
Fragmentos por tipo:           {'politicas': 49, 'credito': 18, 'producto': 15, 'terminos': 7}
Preguntas frecuentes (un fragmento cada una): 24


In [3]:
faq = next(c for c in chunks if c.metadata["es_faq"])
print("— Un fragmento de política: la ruta de encabezados viaja dentro del texto —\n")
print(faq.page_content)
print("\nmetadatos:", {k: faq.metadata[k] for k in ("seccion", "area", "es_faq", "idioma")})

ficha = next(c for c in chunks if c.metadata.get("sku") == "P-106")
print("\n— Una ficha en inglés, tal como la subió el vendedor —\n")
print(ficha.page_content)

— Un fragmento de política: la ruta de encabezados viaja dentro del texto —

Devoluciones y garantías › Preguntas frecuentes › ¿Puedo devolver un producto que ya abrí?

Depende de la categoría. Un producto retornable, como una botella térmica o unas bolsas de algodón,
se puede devolver dentro de los 30 días si está sin uso y con su empaque original. Los productos de
higiene personal abiertos no se pueden devolver por razones sanitarias, salvo que hayan llegado
defectuosos o dañados.

metadatos: {'seccion': 'Devoluciones y garantías › Preguntas frecuentes › ¿Puedo devolver un producto que ya abrí?', 'area': 'devoluciones', 'es_faq': True, 'idioma': 'es'}

— Una ficha en inglés, tal como la subió el vendedor —

Product: Stainless steel thermal bottle 750 ml.
Category: cocina.
Returnable: yes.
Description: Double-wall vacuum insulated bottle that keeps drinks cold for 24 hours and hot for 12 hours. Leak-proof lid with a bamboo cap.
Materials: Food-grade 18/8 stainless steel; bamboo cap wi

In [4]:
partes = componentes.armar_almacen(backend=BACKEND)
almacen = partes.almacen
print(
    f"Base vectorial: {almacen.backend} · tabla {almacen.tabla} · {len(almacen.ids())} fragmentos"
)
print("Por tipo:", almacen.conteo_por_tipo())

Base vectorial: memoria · tabla kb_fake_256 · 89 fragmentos
Por tipo: {'politicas': 49, 'credito': 18, 'producto': 15, 'terminos': 7}


## 2. Una consulta con el contexto a la vista

La pregunta se convierte en vector con el **mismo modelo** que indexó, la base devuelve los *k*
fragmentos más cercanos con su similitud, el prompt los pega con su fuente y el LLM responde citando.
Después, el validador revisa la respuesta en código.

In [5]:
llm = componentes.armar_llm()
r = chains.responder("¿Qué hago con los envases de vidrio vacíos?", almacen, llm)

print(
    f"k={r.recuperacion.k} · umbral={r.recuperacion.umbral} · mejor={r.recuperacion.mejor_score}\n"
)
for i, c in enumerate(r.recuperacion.chunks, 1):
    print(f"{i}. {c.similitud:.4f}  {c.fuente}  [{c.idioma}]")
print("\n— Último mensaje del prompt (el que lleva el contexto) —\n")
print(r.prompt[-1]["contenido"][:1500])

k=4 · umbral=0.0 · mejor=0.5449

1. 0.5449  Sostenibilidad › Programa de reciclaje de envases  [es]
2. 0.4148  Sostenibilidad › Preguntas frecuentes › ¿Qué hago con los envases vacíos?  [es]
3. 0.3273  Sostenibilidad › Preguntas frecuentes › ¿Puedo devolver envases de productos que no compré en EcoMarket?  [es]
4. 0.2582  Catálogo › P-108 · Vaso de vidrio con tapa de bambú 500 ml  [es]

— Último mensaje del prompt (el que lleva el contexto) —

<contexto>
<fragmento fuente="Sostenibilidad › Programa de reciclaje de envases" idioma="es">
Sostenibilidad › Programa de reciclaje de envases

Los clientes pueden devolver los envases vacíos de vidrio y de los productos a granel comprados en
EcoMarket. Los envases se entregan al transportador en la siguiente entrega o en un punto aliado, y
EcoMarket los limpia y los reutiliza o los envía a reciclaje certificado. Por cada 5 envases devueltos,
el cliente recibe un cupón del 10% de descuento para su siguiente compra.
</fragmento>
<fragmento fuente

In [6]:
print("Respuesta:", r.respuesta, "\n")
for nombre, ok, detalle in r.checks:
    print(("✓" if ok else "✗"), nombre, "—", detalle)

Respuesta: Los clientes pueden devolver los envases vacíos de vidrio y de los productos a granel comprados en EcoMarket. [Sostenibilidad › Programa de reciclaje de envases] 

✓ cita_una_fuente — 1 cita(s) válidas
✓ sin_datos_de_pedidos — ok
✓ respuesta_completa — ok


## 3. Abstención: el umbral se calibra, no se adivina

Antes de llamar al LLM se mira la similitud del mejor fragmento. La calibración busca el umbral que
mejor separa los casos dentro y fuera de alcance del set de evaluación, sin dejar sin respuesta a los
translingües (D = dentro, F = fuera, T = translingüe).

In [7]:
grupos = evaluation.mejores_scores(almacen)
cal = evaluation.calibrar_umbral(almacen.perfil, grupos)
print("\n".join(evaluation.histograma(cal)))
print(f"\nUmbral propuesto: {cal.umbral} · exactitud {cal.exactitud}")
if cal.nota:
    print(cal.nota)

0.10-0.15 | F
0.15-0.20 | DDFFT ← umbral
0.20-0.25 | DDDFFTTT
0.25-0.30 | DDDDDDD
0.30-0.35 | DDDDDDD
0.35-0.40 | DDDD
0.40-0.45 | DF
0.45-0.50 | DDDD
0.50-0.55 | DD
0.60-0.65 | DD

Umbral propuesto: 0.1811 · exactitud 0.895
Algún caso translingüe puntúa por debajo de un caso fuera de alcance: no existe un umbral que separe todo. Se prioriza no abstenerse en los translingües.


In [8]:
fuera = chains.responder(
    "¿Quién ganó el partido de fútbol de anoche?", almacen, llm, umbral=cal.umbral
)
print(f"mejor score {fuera.recuperacion.mejor_score} frente al umbral {cal.umbral}")
print("→", fuera.pasos()["2_decision"])
print("Respuesta:", fuera.respuesta)
print("Llamadas al LLM:", fuera.llamadas)

mejor score 0.1796 frente al umbral 0.1811
→ sin evidencia: abstención desde código (no se llamó al LLM)
Respuesta: No tengo información sobre eso en los documentos de EcoMarket. Puedo ayudarle con el estado de su pedido, devoluciones, envíos, pagos, la financiación en cuotas y nuestros productos. Si lo prefiere, le comunico con un agente humano.
Llamadas al LLM: 0


## 4. Integración con el Taller 1

`atender` es el punto de entrada único. Los pedidos se consultan con la cadena del Taller 1 (sin RAG
y sin gastar la llamada del router); las devoluciones usan la cadena del Taller 1 con la política
**recuperada**, y la decisión sigue saliendo de `evaluar_devolucion()`; el crédito y las consultas
generales usan el RAG.

In [9]:
llm_router = componentes.armar_llm(model_role="router")
cliente_t1 = componentes.armar_cliente_t1()
mensajes = [
    "¿Dónde está mi pedido ECO-1005?",
    "Mi pedido ECO-1004 ya llegó, ¿cuánto tiempo tengo para devolver la botella térmica?",
    "Si compro 650.000 pesos a 6 cuotas, ¿cuánto me queda cada cuota?",
    "¿Puedo pagar contra entrega?",
    "¿Cómo preparo un pan de banano?",
]
for mensaje in mensajes:
    a = atencion.atender(mensaje, almacen, llm, cliente_t1, llm_router)
    ok = sum(1 for _, v, _ in a.checks if v)
    print(
        f"■ {mensaje}\n  ruta: {a.ruta} ({a.clasificacion.origen}) · llamadas: {a.llamadas} · "
        f"chequeos {ok}/{len(a.checks)}"
    )
    if a.ruta == "devolucion":
        regla = a.pasos["2_regla_en_codigo"]
        print(f"  decisión en código: {regla['elegible']} ({regla['regla_aplicada']})")
    print(f"  respuesta: {a.respuesta[:160]}\n")

■ ¿Dónde está mi pedido ECO-1005?
  ruta: estado_pedido (tracking) · llamadas: 2 · chequeos 4/5
  respuesta: (respuesta simulada) Su pedido está en estado retrasado. Puede rastrearlo aquí: https://ecomarket.example/rastreo/ECO-1005

■ Mi pedido ECO-1004 ya llegó, ¿cuánto tiempo tengo para devolver la botella térmica?
  ruta: devolucion (tracking) · llamadas: 2 · chequeos 1/2
  decisión en código: si (retornable_en_plazo)
  respuesta: (respuesta simulada) Decisión: si. Producto retornable y dentro del plazo (5 de 30 días desde la entrega).

■ Si compro 650.000 pesos a 6 cuotas, ¿cuánto me queda cada cuota?
  ruta: credito (llm) · llamadas: 2 · chequeos 5/6
  respuesta: Tema: Número de cuotas según el monto. [Reglas de crédito › CR-06 · Número de cuotas según el monto]

■ ¿Puedo pagar contra entrega?
  ruta: consulta_general (llm) · llamadas: 2 · chequeos 3/3
  respuesta: No. [Pagos y cuenta › Preguntas frecuentes › ¿Puedo pagar contra entrega?]

■ ¿Cómo preparo un pan de banano?
  ruta:

## 5. Resumen de los experimentos

Los experimentos con modelos reales se corren con `taller02 evaluar` y dejan su evidencia en
`outputs/`. Esta celda solo lista lo que haya; las cifras se leen en esos archivos y en los
documentos de las fases.

In [10]:
from taller02_rag.config import OUTPUTS_DIR

reportes = sorted(p.name for p in OUTPUTS_DIR.glob("*.md"))
if not reportes:
    print("Todavía no hay corridas reales en outputs/.")
for nombre in reportes:
    print("·", nombre)

Todavía no hay corridas reales en outputs/.
